# Load NN Normal Dataset

In [55]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import numpy as np
np.set_printoptions(legacy='1.25')

from imblearn.over_sampling import SMOTENC
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

RANDOM_STATE = 1

PROJECT_ROOT = Path("..")
INPUT_DIR = PROJECT_ROOT / "data" / "pre-processed"
OUTPUT_DIR = PROJECT_ROOT / "data" / "pre-processed"

nn_normal_df = pd.read_csv(INPUT_DIR / "loan_data_preprocessed.csv")

print("NN normal data shape :", nn_normal_df.shape)
nn_normal_df.head()

NN normal data shape : (44993, 13)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
0,22.0,female,4,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,1
1,21.0,female,1,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,0
2,25.0,female,1,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,1
3,23.0,female,3,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,1
4,24.0,male,4,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,1


# Train Test Split

In [56]:
nn_normal_train_df, nn_normal_test_df = train_test_split(nn_normal_df, test_size=0.2, stratify=nn_normal_df["loan_status"], random_state=RANDOM_STATE)

print(f"NN train data shape: {nn_normal_train_df.shape}")
print(f"NN test data shape: {nn_normal_test_df.shape}")

NN train data shape: (35994, 13)
NN test data shape: (8999, 13)


# Normalization preparation

SMOTENC must run before one-hot encoding. This section:

* normalizes continuous numeric features;
* keeps education as 1–5;
* temporarily ordinal-encodes the nominal text columns;
* prepares the data for SMOTENC.

In [57]:
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder

target = "loan_status"
education_col = "person_education"
nn_normal_X_train_raw_df = nn_normal_train_df.drop(columns=[target]).reset_index(drop=True)
nn_normal_y_train_df = nn_normal_train_df[target].reset_index(drop=True)
nn_normal_X_test_raw_df = nn_normal_test_df.drop(columns=[target]).reset_index(drop=True)
nn_normal_y_test_df = nn_normal_test_df[target].reset_index(drop=True)
categorical_cols = nn_normal_X_train_raw_df.select_dtypes(include="object").columns.tolist()
continuous_cols = nn_normal_X_train_raw_df.select_dtypes(exclude="object").columns.tolist()
continuous_cols.remove(education_col)

print("Continuous columns:", continuous_cols)
print("Ordinal column:", education_col)
print("Categorical columns:", categorical_cols)


continuous_scaler = StandardScaler()
X_train_continuous = continuous_scaler.fit_transform(
    nn_normal_X_train_raw_df[continuous_cols]
)
X_test_continuous = continuous_scaler.transform(
    nn_normal_X_test_raw_df[continuous_cols]
)

# Temporarily encode the categorical columns
# These encoded values are used only by SMOTENC. They are converted to one-hot values later.
category_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)
X_train_categorical = category_encoder.fit_transform(
    nn_normal_X_train_raw_df[categorical_cols]
)
X_test_categorical = category_encoder.transform(
    nn_normal_X_test_raw_df[categorical_cols]
)

# Combine the data for SMOTENC
# Education remains in its original 1–5 form at this stage.
X_train_education = nn_normal_X_train_raw_df[[education_col]].to_numpy()
X_test_education = nn_normal_X_test_raw_df[[education_col]].to_numpy()
smotenc_feature_names = continuous_cols + [education_col] + categorical_cols
X_train_for_smotenc = np.hstack([
    X_train_continuous,
    X_train_education,
    X_train_categorical
])
X_test_for_smotenc = np.hstack([
    X_test_continuous,
    X_test_education,
    X_test_categorical
])

print("Training data prepared for SMOTENC:", X_train_for_smotenc.shape)
print("Test data prepared for SMOTENC:", X_test_for_smotenc.shape)



Continuous columns: ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']
Ordinal column: person_education
Categorical columns: ['person_gender', 'person_home_ownership', 'loan_intent']
Training data prepared for SMOTENC: (35994, 12)
Test data prepared for SMOTENC: (8999, 12)


/var/folders/rf/t52_tqh94zzbcs4g3_xbghnh0000gn/T/ipykernel_26327/1593974819.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = nn_normal_X_train_raw_df.select_dtypes(include="object").columns.tolist()


# Apply SMOTENC
The categorical indices include:

* person_education;
* person_gender;
* person_home_ownership;
* loan_intent.

This means SMOTENC will not interpolate these columns.

In [58]:
first_categorical_index = len(continuous_cols)

categorical_indices = list(
    range(first_categorical_index, len(smotenc_feature_names))
)

smotenc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy="auto",
    k_neighbors=5,
    random_state=RANDOM_STATE
)

X_train_smotenc, y_train_smotenc = smotenc.fit_resample(
    X_train_for_smotenc,
    nn_normal_y_train_df
)

X_train_smotenc_df = pd.DataFrame(
    X_train_smotenc,
    columns=smotenc_feature_names
)

nn_normal_y_train_smote_df = pd.Series(
    y_train_smotenc,
    name=target
)

print("Training shape before SMOTENC:", X_train_for_smotenc.shape)
print("Training shape after SMOTENC:", X_train_smotenc_df.shape)

print("\nClass distribution before SMOTENC:")
print(nn_normal_y_train_df.value_counts())

print("\nClass distribution after SMOTENC:")
print(nn_normal_y_train_smote_df.value_counts())



# Check to ensure education remains an exact level
X_train_smotenc_df[education_col] = (
    X_train_smotenc_df[education_col]
    .round()
    .clip(1, 5)
    .astype(int)
)
print("Education values after SMOTENC:")
print(sorted(X_train_smotenc_df[education_col].unique()))

Training shape before SMOTENC: (35994, 12)
Training shape after SMOTENC: (55988, 12)

Class distribution before SMOTENC:
loan_status
0    27994
1     8000
Name: count, dtype: int64

Class distribution after SMOTENC:
loan_status
0    27994
1    27994
Name: count, dtype: int64
Education values after SMOTENC:
[1, 2, 3, 4, 5]


# Convert categories to one-hot columns

In [59]:
# Fit the one-hot encoder using the original training data.
onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
onehot_encoder.fit(
    nn_normal_X_train_raw_df[categorical_cols]
)

# Convert the temporary SMOTENC category codes back to their original text labels:
X_train_smotenc_df[categorical_cols] = (
    X_train_smotenc_df[categorical_cols].round()
)
smotenc_categories = category_encoder.inverse_transform(
    X_train_smotenc_df[categorical_cols]
)
smotenc_categories_df = pd.DataFrame(
    smotenc_categories,
    columns=categorical_cols
)

# Create one-hot columns for the SMOTENC training data and untouched test data:
X_train_onehot = onehot_encoder.transform(
    smotenc_categories_df
)
X_test_onehot = onehot_encoder.transform(
    nn_normal_X_test_raw_df[categorical_cols]
)
onehot_feature_names = onehot_encoder.get_feature_names_out(
    categorical_cols
).tolist()

# Normalize education after SMOTENC

Education is now guaranteed to be one of 1–5

In [60]:
# Keep education as exact ordinal values from 1 to 5
X_train_smotenc_df[education_col] = (
    X_train_smotenc_df[education_col]
    .round()
    .clip(1, 5)
    .astype(int)
)
X_train_education = X_train_smotenc_df[[education_col]].to_numpy()
X_test_education = (
    nn_normal_X_test_raw_df[[education_col]]
    .round()
    .clip(1, 5)
    .astype(int)
    .to_numpy()
)
X_train_education_original = (
    nn_normal_X_train_raw_df[[education_col]]
    .round()
    .clip(1, 5)
    .astype(int)
    .to_numpy()
)

print("Education values after SMOTENC:")
print(sorted(np.unique(X_train_education)))
print("Education values in the test set:")
print(sorted(np.unique(X_test_education)))

Education values after SMOTENC:
[1, 2, 3, 4, 5]
Education values in the test set:
[1, 2, 3, 4, 5]


# Create the final neural-network datasets

In [61]:
# Final feature names
nn_feature_names = continuous_cols + [education_col] + onehot_feature_names

# SMOTENC normalized training data
nn_normal_X_train_smote = np.hstack([
    X_train_smotenc_df[continuous_cols].to_numpy(),
    X_train_education,
    X_train_onehot
])
nn_normal_X_train_smote_df = pd.DataFrame(
    nn_normal_X_train_smote,
    columns=nn_feature_names
)
print("Final SMOTENC training shape:", nn_normal_X_train_smote_df.shape)

# Untouched normalized test data
nn_normal_X_test = np.hstack([
    X_test_continuous,
    X_test_education,
    X_test_onehot
])
nn_normal_X_test_df = pd.DataFrame(
    nn_normal_X_test,
    columns=nn_feature_names
)
print("Final test shape:", nn_normal_X_test_df.shape)

# Normalized training set without SMOTENC
X_train_original_onehot = onehot_encoder.transform(
    nn_normal_X_train_raw_df[categorical_cols]
)
nn_normal_X_train = np.hstack([
    X_train_continuous,
    X_train_education_original,
    X_train_original_onehot
])
nn_normal_X_train_df = pd.DataFrame(
    nn_normal_X_train,
    columns=nn_feature_names
)
print("Normalized training shape without SMOTENC:", nn_normal_X_train_df.shape)

# Convert education back to integer
# np.hstack() combines integer and floating-point arrays, NumPy converts everything to floats. Therefore, the CSV may show education as 1.0, 2.0, ...
nn_normal_X_train_smote_df[education_col] = (
    nn_normal_X_train_smote_df[education_col].astype(int)
)
nn_normal_X_train_df[education_col] = (
    nn_normal_X_train_df[education_col].astype(int)
)
nn_normal_X_test_df[education_col] = (
    nn_normal_X_test_df[education_col].astype(int)
)

# Check categorical values
print("Education values in SMOTENC training data:")
print(sorted(nn_normal_X_train_smote_df[education_col].unique()))

print("\nEducation values in original training data:")
print(sorted(nn_normal_X_train_df[education_col].unique()))

print("\nEducation values in test data:")
print(sorted(nn_normal_X_test_df[education_col].unique()))

print("\nUnique values in one-hot columns:")
print(np.unique(nn_normal_X_train_smote_df[onehot_feature_names].to_numpy()))

Final SMOTENC training shape: (55988, 21)
Final test shape: (8999, 21)
Normalized training shape without SMOTENC: (35994, 21)
Education values in SMOTENC training data:
[1, 2, 3, 4, 5]

Education values in original training data:
[1, 2, 3, 4, 5]

Education values in test data:
[1, 2, 3, 4, 5]

Unique values in one-hot columns:
[0. 1.]


# Save Outputs

In [62]:
nn_normal_train_smote_output_path = OUTPUT_DIR / "loan_data_nn_normal_train_smote.csv"
nn_normal_train_output_path = OUTPUT_DIR / "loan_data_nn_normal_train.csv"
nn_normal_test_output_path = OUTPUT_DIR / "loan_data_nn_normal_test.csv"

nn_normal_train_smote_df = pd.concat([
    nn_normal_X_train_smote_df.reset_index(drop=True),
    nn_normal_y_train_smote_df.reset_index(drop=True)
], axis=1)

nn_normal_train_df = pd.concat([
    nn_normal_X_train_df.reset_index(drop=True),
    nn_normal_y_train_df.reset_index(drop=True)
], axis=1)

nn_normal_test_processed_df = pd.concat([
    nn_normal_X_test_df.reset_index(drop=True),
    nn_normal_y_test_df.reset_index(drop=True)
], axis=1)

nn_normal_train_smote_df.to_csv(
    nn_normal_train_smote_output_path,
    index=False
)

nn_normal_train_df.to_csv(
    nn_normal_train_output_path,
    index=False
)

nn_normal_test_processed_df.to_csv(
    nn_normal_test_output_path,
    index=False
)

print("Saved SMOTENC training data to:", nn_normal_train_smote_output_path)
print("Saved normalized training data to:", nn_normal_train_output_path)
print("Saved normalized test data to:", nn_normal_test_output_path)

Saved SMOTENC training data to: ../data/pre-processed/loan_data_nn_normal_train_smote.csv
Saved normalized training data to: ../data/pre-processed/loan_data_nn_normal_train.csv
Saved normalized test data to: ../data/pre-processed/loan_data_nn_normal_test.csv
